# 02 — CNN-BiLSTM Model Training

Data preparation, feature engineering, augmentation, arsitektur model, training, dan studi ablasi.

In [ ]:
# KODE TRAINING DAN STUDI ABLASI CNN-LSTM

import os, random
import numpy as np
import tensorflow as tf
from scipy.interpolate import CubicSpline
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, LSTM,
                                     Dense, Dropout, BatchNormalization, Activation, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import pandas as pd


#SEED untuk memastikan semua proses menghasilkan urutan yang sama setiap kali kode dijalankan
#Penting agar hasil eksperimen dapat direproduksi dan perbandingan antar skenario ablasi menjadi adil
SEED_VALUE = 42
os.environ['PYTHONHASHSEED'] = str(SEED_VALUE)
random.seed(SEED_VALUE)
np.random.seed(SEED_VALUE)
tf.random.set_seed(SEED_VALUE)
print("Seed 42 terkunci")

# KONFIGURASI FOLDER
PATH_DATASET_ROOT = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/Dataset_NPY_Final_45Fitur'
PATH_SAVE_ROOT    = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/Hasil_Ablation_Murni_Final'
os.makedirs(PATH_SAVE_ROOT, exist_ok=True)

CLASSES_LIST = ["jumpingjack", "lunges", "pushups", "squat", "Situp", "idle"]
NUM_CLASSES  = len(CLASSES_LIST)
SEQ_LEN      = 30
EPOCHS       = 100
BATCH_SIZE   = 32


#AUGMENTASI DATA (Sebelum Fitur Engineering)
def aug_flip(raw):
    f = raw.copy()
    for j in range(12):
        f[:, j*3] = 1.0 - f[:, j*3]
    COORD_FLIP_PAIRS = [(0,1),(2,3),(4,5),(6,7),(8,9),(10,11)]
    for li, ri in COORD_FLIP_PAIRS:
        lc = slice(li*3, li*3+3)
        rc = slice(ri*3, ri*3+3)
        f[:, lc], f[:, rc] = f[:, rc].copy(), f[:, lc].copy()
    f[:, 38], f[:, 39] = raw[:, 39].copy(), raw[:, 38].copy()
    f[:, 40], f[:, 41] = raw[:, 41].copy(), raw[:, 40].copy()
    return f

def time_warp(sequence, sigma=0.2):
    orig_steps   = np.arange(sequence.shape[0])
    knot_steps   = np.linspace(0, sequence.shape[0] - 1, num=6)
    random_warps = np.random.normal(loc=1.0, scale=sigma, size=(6,))
    cs           = CubicSpline(knot_steps, random_warps)
    warp_curve   = cs(orig_steps)
    new_steps    = np.cumsum(warp_curve)
    new_steps    = new_steps - new_steps[0]
    new_steps    = new_steps / new_steps[-1] * (sequence.shape[0] - 1)
    ret = np.zeros_like(sequence)
    for dim in range(sequence.shape[1]):
        ret[:, dim] = np.interp(orig_steps, new_steps, sequence[:, dim])
    return ret

def augment_45(window_45, class_name):
    if class_name == 'idle':
        return window_45 + np.random.normal(0, 0.002, window_45.shape)
    aug = window_45.copy()
    if np.random.rand() < 0.5: aug = aug_flip(aug)
    if np.random.rand() < 0.5: aug = time_warp(aug)
    if np.random.rand() < 0.5: aug += np.random.normal(0, 0.005, aug.shape)
    return aug

#FEATURE ENGINEERING (KINEMATIKA)
def compute_kinematics(window, smooth_sigma=0.3):
    smoothed = np.zeros_like(window)
    for j in range(window.shape[1]):
        smoothed[:, j] = gaussian_filter1d(window[:, j], sigma=smooth_sigma)
    vel = np.diff(smoothed, axis=0, prepend=smoothed[0:1])
    acc = np.diff(vel,      axis=0, prepend=vel[0:1])
    return np.concatenate([window, vel, acc], axis=1)


#LOAD DATA
def load_data(mode, use_raw36=False, use_fe=True, use_aug=False):
    features, labels = [], []
    class_map  = {c: i for i, c in enumerate(CLASSES_LIST)}
    split_path = os.path.join(PATH_DATASET_ROOT, mode)
    stride     = SEQ_LEN // 2 if mode == 'train' else SEQ_LEN #stride: 15 train (overlap 50%), 30 test()

    print(f"\n[LOAD] mode={mode} | raw36={use_raw36} | FE={use_fe} | Aug={use_aug} | stride={stride}")

    for class_name in CLASSES_LIST:
        class_dir = os.path.join(split_path, class_name)
        if not os.path.exists(class_dir):
            continue

        # FIX SORTED: Agar urutan data dibaca sama terus
        files = [os.path.join(class_dir, f) for f in sorted(os.listdir(class_dir)) if f.endswith('.npy')]
        win_count = 0

        for fpath in files:
            data = np.load(fpath)
            if data.shape[0] < SEQ_LEN:
                continue

            for i in range(0, data.shape[0] - SEQ_LEN + 1, stride):
                if use_raw36:
                    window = data[i: i+SEQ_LEN, :36].copy()
                else:
                    window = data[i: i+SEQ_LEN, :45].copy()

                if use_aug and mode == 'train':
                    aug_win = augment_45(
                        window if not use_raw36 else np.concatenate([window, np.zeros((SEQ_LEN, 9))], axis=1),
                        class_name)
                    aug_win = aug_win if not use_raw36 else aug_win[:, :36]
                    aug_feat = compute_kinematics(aug_win) if use_fe else aug_win
                    features.append(aug_feat)
                    labels.append(class_map[class_name])

                feat = compute_kinematics(window) if use_fe else window
                features.append(feat)
                labels.append(class_map[class_name])
                win_count += 1

        print(f"  {class_name}: {len(files)} file → {win_count} windows")

    X = np.array(features, dtype=np.float32)
    y = np.array(labels,   dtype=np.int32)
    print(f"  TOTAL: X={X.shape} | y={y.shape}")
    return X, y


# NORMALISASI DATA (ROBUST SCALING)
def normalize(X_train, X_val, X_test):
    median = np.median(X_train, axis=(0, 1), keepdims=True)
    q75    = np.percentile(X_train, 75, axis=(0, 1))
    q25    = np.percentile(X_train, 25, axis=(0, 1))
    iqr    = (q75 - q25) + 1e-8
    X_tr   = (X_train - median) / iqr
    X_v    = (X_val   - median) / iqr
    X_te   = (X_test  - median) / iqr
    return X_tr, X_v, X_te, median, iqr


# ARSITEKTUR
def build_cnn_only(input_shape, num_classes):
    inp = Input(shape=input_shape)
    x   = Conv1D(128, 5, padding='same', kernel_regularizer=l2(1e-4))(inp)
    x   = BatchNormalization()(x); x = Activation('relu')(x)
    x   = MaxPooling1D(2)(x);      x = Dropout(0.3)(x)
    x   = Conv1D(256, 3, padding='same', kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Activation('relu')(x)
    x   = GlobalAveragePooling1D()(x); x = Dropout(0.3)(x)
    x   = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x);     x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out, name='CNN_Only')

def build_lstm_only(input_shape, num_classes):
    inp = Input(shape=input_shape)
    x   = LSTM(256, return_sequences=True, kernel_regularizer=l2(1e-4))(inp)
    x   = BatchNormalization()(x); x = Dropout(0.3)(x)
    x   = LSTM(128, return_sequences=False, kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Dropout(0.3)(x)
    x   = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out, name='LSTM_Only')

def build_cnn_lstm(input_shape, num_classes):
    inp = Input(shape=input_shape)
    x   = Conv1D(128, 5, padding='same', kernel_regularizer=l2(1e-4))(inp)
    x   = BatchNormalization()(x); x = Activation('relu')(x)
    x   = MaxPooling1D(2)(x);      x = Dropout(0.3)(x)
    x   = Conv1D(256, 3, padding='same', kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Activation('relu')(x)
    x   = MaxPooling1D(2)(x);      x = Dropout(0.3)(x)
    x   = LSTM(256, return_sequences=True, kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Dropout(0.3)(x)
    x   = LSTM(128, return_sequences=False, kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Dropout(0.3)(x)
    x   = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out, name='CNN_LSTM')

#TEMPORAL ENSEMBLE
def temporal_ensemble(y_pred_probs, window_size=5):
    smoothed = np.copy(y_pred_probs)
    for i in range(len(y_pred_probs)):
        start = max(0, i - window_size + 1)
        smoothed[i] = np.mean(y_pred_probs[start:i+1], axis=0)
    return np.argmax(smoothed, axis=1)

#PLOT/VISUALISASI
def plot_results(history, y_test, y_pred_raw, y_pred_ens, case_id, case_name, acc_raw, acc_ens, save_dir):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history.history['accuracy'],     label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Val')
    axes[0].set_title(f'K{case_id} Accuracy\nRaw={acc_raw:.2f}% | Ens={acc_ens:.2f}%')
    axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(history.history['loss'],     label='Train')
    axes[1].plot(history.history['val_loss'], label='Val')
    axes[1].set_title(f'K{case_id} Loss')
    axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)

    cm = confusion_matrix(y_test, y_pred_ens)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
                xticklabels=CLASSES_LIST, yticklabels=CLASSES_LIST)
    axes[2].set_title(f'K{case_id} CM (Ensemble: {acc_ens:.2f}%)')
    axes[2].set_ylabel('Aktual'); axes[2].set_xlabel('Prediksi')

    plt.tight_layout()
    path = os.path.join(save_dir, f'kasus{case_id}_murni_plot.png')
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"  [SAVED] Plot → {path}")


#TRAIN PER KASUS
def train_case(case_id, case_name, build_fn, use_raw36, use_fe, use_aug):
    print(f"\n{'='*65}")
    print(f"  MEMULAI KASUS {case_id}: {case_name}")
    print(f"{'='*65}")

    save_dir = os.path.join(PATH_SAVE_ROOT, f'kasus{case_id}')
    os.makedirs(save_dir, exist_ok=True)
    model_path = os.path.join(save_dir, f'kasus{case_id}_murni_best.keras')
  #LOAD DATA
    X_tr_raw, y_tr = load_data('train', use_raw36, use_fe, use_aug)
    X_te_raw, y_te = load_data('test',  use_raw36, use_fe, False)

    if len(X_tr_raw) == 0:
        print(f"  [ERROR] Dataset kosong!")
        return None
  #SPLIT TRAIN-VALIDATION (85:15)
    X_tr, X_val, y_tr_s, y_val_s = train_test_split(
        X_tr_raw, y_tr, test_size=0.15, random_state=SEED_VALUE, stratify=y_tr)
  #NORMALISASI DATA
    X_tr, X_val, X_te, median, iqr = normalize(X_tr, X_val, X_te_raw)
    print(f"  [INFO] Train={len(X_tr)} | Val={len(X_val)} | Test={len(X_te)}")

    if case_id in [6, 7]:
        np.save(os.path.join(save_dir, 'scaler_median.npy'), median)
        np.save(os.path.join(save_dir, 'scaler_iqr.npy'),    iqr)
        print(f"  [SAVED] Scaler (Penting buat Real-Time) tersimpan di → {save_dir}")
  #CLASS WEIGHTING MENGATASI KETIDAKSEIMBANGAN DATA
    weights = compute_class_weight('balanced', classes=np.unique(y_tr_s), y=y_tr_s)
    cw_dict = dict(enumerate(weights))
  #One-hot encoding
    y_tr_cat  = to_categorical(y_tr_s,  NUM_CLASSES)
    y_val_cat = to_categorical(y_val_s, NUM_CLASSES)
    y_te_cat  = to_categorical(y_te,    NUM_CLASSES)

    # FIX CLEAR SESSION: Bersihkan RAM GPU dari model sebelumnya
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED_VALUE)
    np.random.seed(SEED_VALUE)

    #build model
    input_shape = (SEQ_LEN, X_tr.shape[2])
    model = build_fn(input_shape, NUM_CLASSES)

    model.compile(optimizer=AdamW(learning_rate=0.001, weight_decay=1e-4),
                  loss='categorical_crossentropy', metrics=['accuracy'])

    cb_list = [
        ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1),
    ]
    #TRAINING
    history = model.fit(
        X_tr, y_tr_cat, validation_data=(X_val, y_val_cat),
        epochs=EPOCHS, batch_size=BATCH_SIZE, class_weight=cw_dict,
        callbacks=cb_list, verbose=1)
    #EVALUASI
    best_model   = tf.keras.models.load_model(model_path)
    y_pred_probs = best_model.predict(X_te, verbose=0)
    y_pred_raw   = np.argmax(y_pred_probs, axis=1)
    y_pred_ens   = temporal_ensemble(y_pred_probs, window_size=5)

    acc_raw = np.mean(y_pred_raw == y_te) * 100
    acc_ens = np.mean(y_pred_ens == y_te) * 100

    print(f"\n  [RESULT] Kasus {case_id} — Raw: {acc_raw:.2f}% | Ensemble: {acc_ens:.2f}%")
    print(classification_report(y_te, y_pred_ens, target_names=CLASSES_LIST))

    pd.DataFrame(history.history).to_csv(os.path.join(save_dir, f'kasus{case_id}_murni_history.csv'), index=False)
    plot_results(history, y_te, y_pred_raw, y_pred_ens, case_id, case_name, acc_raw, acc_ens, save_dir)

    return {'kasus': case_id, 'nama': case_name, 'raw36': use_raw36, 'FE': use_fe, 'Aug': use_aug,
            'n_feat': X_tr.shape[2], 'acc_raw': f'{acc_raw:.2f}%', 'acc_ensemble': f'{acc_ens:.2f}%'}

#MAIN
if __name__ == '__main__':

    # Pilih kasus yang akan dijalankan (1-7)
    # Untuk training model final, gunakan 7
    TARGET_KASUS = 7

    ABLATION_CASES = [
        (1, 'CNN-Only  | 36 raw + FE = 108 fitur | No Aug', build_cnn_only,  True,  True, False),
        (2, 'LSTM-Only | 36 raw + FE = 108 fitur | No Aug', build_lstm_only, True,  True, False),
        (3, 'CNN-LSTM  | 36 raw + FE = 108 fitur | No Aug', build_cnn_lstm,  True,  True, False),
        (4, 'CNN-Only  | 45 FE = 135 fitur | No Aug',       build_cnn_only,  False, True,  False),
        (5, 'LSTM-Only | 45 FE = 135 fitur | No Aug',       build_lstm_only, False, True,  False),
        (6, 'CNN-LSTM  | 45 FE = 135 fitur | No Aug',       build_cnn_lstm,  False, True,  False),
        (7, 'CNN-LSTM  | 45 FE = 135 fitur | Aug',          build_cnn_lstm,  False, True,  True),
    ]

    print(f"\n{'='*65}")
    print(f"MENJALANKAN MODE MURNI UNTUK KASUS {TARGET_KASUS}")
    print(f"{'='*65}")

    for case_data in ABLATION_CASES:
        if case_data[0] == TARGET_KASUS:
            cid, cname, bfn, ur36, ufe, uaug = case_data
            result = train_case(cid, cname, bfn, ur36, ufe, uaug)
            print(f"\n EKSEKUSI KASUS {TARGET_KASUS} SELESAI!")
            break

## Catatan Hasil K7

Notebook sumber mencatat K7: Raw 89.27% dan temporal ensemble 94.96% pada evaluasi yang tercantum di notebook. Angka headline final perlu diverifikasi dengan hasil final penelitian sebelum dipublikasikan.